# 11 · Where buffalo are seen: an effort-adjusted sightings map

Notebooks 06–10 tried to **model** buffalo habitat and concluded that the data isn't enough. The GBIF sightings are too few and too tied to roads and the visited core for a habitat model to work in unseen areas.

This notebook doesn't model anything. It **describes** where buffalo have actually been seen, 2017–2025, in a way that is fair to places with fewer visitors:

- **Raw count:** average buffalo sightings per year in each 5 km cell. Simple, but busy tourist spots win just because more people look there.
- **Effort-adjusted share:** buffalo sightings ÷ **all** mammal sightings in the same cell. Everyone who logs a lion or a zebra in a cell also had the chance to log a buffalo there. So the share answers: *"of the animals people report here, how many are buffalo?"*
- **Camera traps:** buffalo photos per 100 days a camera was running. This is effort-adjusted by design, but only for the central camera area and for 2010–2013.

All grid cells are built in meters (EPSG:32736, ADR-0001). No Earth Engine is needed.

In [1]:
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import shapely
from shapely.geometry import box

import buffalo_data as bd
from buffalo_data import UTM, BuffaloConfig

config = BuffaloConfig(cache_dir="cache")

SPECIES = {"buffalo": 2441034}
CELL_SIZE_M = 5_000
MIN_MAMMAL_RECORDS = 20

records = bd.fetch_mammal_records(config)
n_years = config.end_year - config.start_year + 1
print(f"{len(records)} mammal sightings, {config.start_year}-{config.end_year} ({n_years} years)")
for name, key in SPECIES.items():
    print(f"  {name}: {(records['speciesKey'] == key).sum()} sightings")

mammal_records: loaded from cache\buffalo_mammal_records_8be5ffc111.parquet
mammal records in box: 15873
15873 mammal sightings, 2017-2025 (9 years)
  buffalo: 705 sightings


## 1. Counts per 5 km cell

Each sighting falls into one 5 km × 5 km cell. Per cell:
- `mammal_records`: all mammal sightings, a measure of how much people looked
- `<species>`: that species' sightings, plus the average per year
- `<species>_share`: the species' share of the cell's mammal sightings, only where there are at least `MIN_MAMMAL_RECORDS`
- a **95 % range** for the share (Wilson interval). With 20 records, a share of 10 % could really be anywhere from about 3 % to 30 %. The range shows how sure each cell's number is.

In [2]:
def wilson_range(k, n, z=1.96):
    """95 % range for a share of k out of n (Wilson score interval)."""
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


x, y = bd.to_utm.transform(records["decimalLongitude"].values, records["decimalLatitude"].values)
records["cx"], records["cy"] = (x // CELL_SIZE_M).astype(int), (y // CELL_SIZE_M).astype(int)

grid = records.groupby(["cx", "cy"]).size().rename("mammal_records").to_frame()
for name, key in SPECIES.items():
    grid[name] = records.assign(hit=records["speciesKey"] == key).groupby(["cx", "cy"])["hit"].sum()
    grid[f"{name}_per_year"] = grid[name] / n_years
    enough = grid["mammal_records"] >= MIN_MAMMAL_RECORDS
    grid[f"{name}_share"] = (grid[name] / grid["mammal_records"]).where(enough)
    grid[f"{name}_low"], grid[f"{name}_high"] = wilson_range(grid[name], grid["mammal_records"])
grid = grid.reset_index()

grid["cell_id"] = grid["cx"].astype(str) + "_" + grid["cy"].astype(str)
squares = gpd.GeoDataFrame(
    grid[["cell_id"]],
    geometry=[box(cx * CELL_SIZE_M, cy * CELL_SIZE_M, (cx + 1) * CELL_SIZE_M, (cy + 1) * CELL_SIZE_M) for cx, cy in zip(grid["cx"], grid["cy"])],
    crs=UTM,
).to_crs("EPSG:4326").set_index("cell_id")
grid["lon"], grid["lat"] = bd.to_lonlat.transform((grid["cx"] + 0.5).values * CELL_SIZE_M, (grid["cy"] + 0.5).values * CELL_SIZE_M)
squares.geometry = shapely.set_precision(squares.geometry, 1e-5)
geojson = json.loads(squares.to_json())

overall = {name: (records["speciesKey"] == key).mean() for name, key in SPECIES.items()}
print(f"cells with any mammal sighting: {len(grid)}, with at least {MIN_MAMMAL_RECORDS}: {(grid['mammal_records'] >= MIN_MAMMAL_RECORDS).sum()}")
for name in SPECIES:
    print(f"{name}: seen in {(grid[name] > 0).sum()} cells; overall share of all mammal sightings {overall[name]:.1%}")

cells with any mammal sighting: 953, with at least 20: 164
buffalo: seen in 261 cells; overall share of all mammal sightings 4.4%


### Where the share is highest

Cells with at least `MIN_MAMMAL_RECORDS` mammal sightings, sorted by buffalo share. Look at the **95 % range** next to the share: a high share with a wide range rests on few records.

In [3]:
name = "buffalo"
top = grid.dropna(subset=[f"{name}_share"]).sort_values(f"{name}_share", ascending=False).head(15)
pd.DataFrame({
    "lon": top["lon"].round(3), "lat": top["lat"].round(3),
    "mammal sightings": top["mammal_records"], f"{name} sightings": top[name],
    f"{name} per year": top[f"{name}_per_year"].round(1),
    "share": (top[f"{name}_share"] * 100).round(1).astype(str) + " %",
    "95 % range": [f"{lo:.0%} to {hi:.0%}" for lo, hi in zip(top[f"{name}_low"], top[f"{name}_high"])],
}).reset_index(drop=True)

,lon,lat,mammal sightings,buffalo sightings,buffalo per year,share,95 % range
0,35.451,-3.005,51,14,1.6,27.5 %,17% to 41%
1,35.496,-3.005,39,9,1.0,23.1 %,13% to 38%
2,35.586,-3.050,54,12,1.3,22.2 %,13% to 35%
3,35.452,-3.186,71,15,1.7,21.1 %,13% to 32%
4,35.497,-3.096,51,10,1.1,19.6 %,11% to 32%
5,35.542,-3.322,31,6,0.7,19.4 %,9% to 36%
6,35.452,-3.096,89,17,1.9,19.1 %,12% to 28%
7,35.452,-3.051,64,12,1.3,18.8 %,11% to 30%
8,35.541,-3.005,43,8,0.9,18.6 %,10% to 33%
9,35.903,-3.546,22,4,0.4,18.2 %,7% to 39%


## 2. Map

Three layers, switched with the dropdown:
- **Buffalo share (effort-adjusted)**, the default. Grey cells have fewer than `MIN_MAMMAL_RECORDS` mammal sightings: people logged too little there to say.
- **Buffalo sightings per year:** the raw average. Compare it with the share to see how much of the raw picture is just "many visitors".
- **Camera traps:** buffalo photos per 100 trap-days. A different period (2010–2013) and only the central area.

Hover over a cell for its counts and 95 % range.

In [4]:
name = "buffalo"
cam = bd.load_camera_sites(config)

BLUES = [[0, "#d6e6fa"], [0.5, "#2a78d6"], [1, "#0d366b"]]
GREY = "#d9d9d9"

shown = grid.dropna(subset=[f"{name}_share"])
too_few = grid[grid[f"{name}_share"].isna()]
hover = (f"{name} sightings: %{{customdata[0]}} (%{{customdata[1]:.1f}} per year)<br>"
         "all mammal sightings: %{customdata[2]}<br>"
         f"{name} share: %{{customdata[3]:.1%}} (95 %% range %{{customdata[4]:.0%}} to %{{customdata[5]:.0%}})<extra></extra>")
custom = lambda d: np.c_[d[name], d[f"{name}_per_year"], d["mammal_records"], d[f"{name}_share"].fillna(0), d[f"{name}_low"], d[f"{name}_high"]]
outline = dict(marker_line_width=0.4, marker_line_color="#ffffff")

traces = [
    go.Choroplethmap(geojson=geojson, locations=too_few["cell_id"], z=np.zeros(len(too_few)), colorscale=[[0, GREY], [1, GREY]],
                     showscale=False, name="too few records", customdata=custom(too_few), hovertemplate=hover, **outline),
    go.Choroplethmap(geojson=geojson, locations=shown["cell_id"], z=shown[f"{name}_share"] * 100, colorscale=BLUES,
                     zmin=0, zmax=float(shown[f"{name}_share"].quantile(0.95) * 100), name=f"{name} share",
                     colorbar=dict(title="share of<br>mammal<br>sightings (%)", thickness=12),
                     customdata=custom(shown), hovertemplate=hover, **outline),
    go.Choroplethmap(geojson=geojson, locations=grid["cell_id"], z=grid[f"{name}_per_year"], colorscale=BLUES,
                     zmin=0, zmax=float(grid[f"{name}_per_year"].quantile(0.99)), name=f"{name} per year", visible=False,
                     colorbar=dict(title="sightings<br>per year", thickness=12),
                     customdata=custom(grid), hovertemplate=hover, **outline),
    go.Scattermap(lat=cam["decimalLatitude"], lon=cam["decimalLongitude"], mode="markers", name="camera traps", visible=False,
                  marker=dict(size=9, color=cam["rate"], colorscale=BLUES, cmin=0, cmax=float(cam["rate"].quantile(0.95)),
                              showscale=True, colorbar=dict(title="photos per<br>100 trap-days", thickness=12)),
                  customdata=np.c_[cam["SiteID"], cam["rate"], cam["trap_days"]],
                  hovertemplate="camera %{customdata[0]}<br>%{customdata[1]:.1f} buffalo photos per 100 trap-days<br>"
                                "%{customdata[2]} trap-days<extra></extra>"),
]
layers = {
    f"{name} share (effort-adjusted)": [True, True, False, False],
    f"{name} sightings per year": [False, False, True, False],
    "camera traps (2010-2013)": [False, False, False, True],
}
west, south, east, north = config.bounds
fig = go.Figure(traces)
fig.update_layout(
    map=dict(style="carto-positron", zoom=7.3, center={"lat": (south + north) / 2, "lon": (west + east) / 2}),
    updatemenus=[dict(buttons=[dict(label=label, method="update", args=[{"visible": visible}, {"title": f"Where {name} are seen: {label}"}])
                               for label, visible in layers.items()], x=0, y=1.08, xanchor="left")],
    title=f"Where {name} are seen: {next(iter(layers))}",
    height=700, margin=dict(t=80, l=0, r=0, b=0), showlegend=False,
)
fig.show()

**How to read it, honestly:**
- **The share compares places, not populations.** 30 % means "30 % of the mammals people logged here were buffalo". It doesn't mean "there are many buffalo here". A place full of wildebeest can have many buffalo and still a low share.
- **The share also reflects what people bother to log.** Visitors may log lions more eagerly than buffalo, but that preference is roughly the same everywhere, so comparing cells is still fair.
- **Grey isn't "no buffalo",** it's "too few records to say". Large grey areas are where the data simply doesn't reach, which is the same gap that limited the models in 06–10.
- **The camera layer** is the only one with real effort and absences, but it's older and covers only the centre. Where it overlaps the GBIF cells, the two should broadly agree.